In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import random


def get_weather(city: str) -> str:
    """
    Use this tool when the user asks about the current weather, 
    temperature, or climate conditions of a specific city.
    
    Args:
        city: The name of the city (e.g., 'London', 'New York').
    """
    temp = random.randint(15, 35)
    return f"The weather in {city} is sunny now. temperature is {temp} degree."

In [4]:
def transportation_info(city: str) -> str:
    """ 
     Use this tool when the user asks about the transportation like bus,
     taxi, metro rail etc of a specific city

     Args:
        city: The name of the city (e.g., 'London', 'New York').
    """
    return f"All the public buses in {city} are unavailable today. Metro rail and taxi are available"

In [5]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    temperature=0.1,
    task="text-generation",
    max_new_tokens=512,
)

chat_model = ChatHuggingFace(llm = llm)

e:\Personal Project\my_first_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [6]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key = groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    max_tokens=1024
)

system_instruction = (
    "You are a helpful assistant. Use your tools to gather real-time data.\n\n"
    "CRITICAL REQUIREMENT:\n"
    "When tools return information, you MUST directly answer the user's question "
    "using that specific data. Combine the findings from all tools into a short, "
    "Do not just output a generic disclaimer."
)

agent = create_agent(
    model=llm,
    tools=[get_weather,transportation_info],
    system_prompt=system_instruction
)

In [7]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "how is transportations situation in Mumbai now and what is the temperature today?"}]},
    config={"recursion_limit": 5} # Cap it at 5 graph transitions max
)
response

{'messages': [HumanMessage(content='how is transportations situation in Mumbai now and what is the temperature today?', additional_kwargs={}, response_metadata={}, id='fc018c51-3c50-4851-b25d-ad100a753659'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qvm6hpags', 'function': {'arguments': '{"city":"Mumbai"}', 'name': 'transportation_info'}, 'type': 'function'}, {'id': 'ssdgktja3', 'function': {'arguments': '{"city":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 496, 'total_tokens': 617, 'completion_time': 0.265002429, 'completion_tokens_details': None, 'prompt_time': 0.042749438, 'prompt_tokens_details': None, 'queue_time': 0.047996391, 'total_time': 0.307751867}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f6970-e7d7-7991-8d

In [ ]:
response = llm.invoke("Who won last fifa world cup?")
print(response.content)

The 2022 FIFA World Cup was won by Argentina, with Lionel Messi being named the tournament's best player. They defeated France 4-2 in a penalty shootout after the match ended 3-3 after extra time in the final on December 18, 2022.


RAG for tool calling in agent

In [8]:
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings 
from dotenv import load_dotenv

load_dotenv()

# 1. Define a pool of specialized tools
@tool
def get_crypto_price(ticker: str) -> str:
    """Get the live price of a cryptocurrency ticker (e.g., BTC, ETH)."""
    return f"The price of {ticker} is $65,000."

@tool
def get_weather(city: str) -> str:
    """Get the current weather forecast for a given city."""
    return f"The weather in {city} is sunny and 72°F."

@tool
def calculate_mortgage(principal: int, rate: float, years: int) -> str:
    """Calculate monthly mortgage payments based on loan details."""
    # Dummy calculation for demonstration
    return f"Your monthly payment for a ${principal} loan is $2,100."

# Keep all available tools in a dictionary mapping their name -> tool object
tool_pool = {
    "get_crypto_price": get_crypto_price,
    "get_weather": get_weather,
    "calculate_mortgage": calculate_mortgage
}

# 2. Build the Tool Index (RAG for Tools)
# We store the tool descriptions in a vector store to search them semantically
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
tool_vector_store = InMemoryVectorStore(embeddings)

# Add tools to our vector store using their descriptions as the index text
for name, tool_obj in tool_pool.items():
    tool_vector_store.add_texts(
        texts=[tool_obj.description],
        metadatas=[{"name": name}]
    )

# 3. Dynamic Retrieval Function
def retrieve_relevant_tools(user_query: str, k: int = 5) -> list:
    """Finds the most contextually appropriate tools for the query."""
    # Search the vector store for descriptions matching the user's intent
    docs = tool_vector_store.similarity_search(user_query, k=k)

    print(docs)
    
    # Map the matched metadata back to the actual tool objects
    selected_tools = [tool_pool[doc.metadata["name"]] for doc in docs]
    return selected_tools



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2447.59it/s]


In [9]:
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_retries=2,
    max_tokens=1024
)

In [10]:
def run_agent_with_retrieved_tools(user_query: str):
    # Tightened the instruction to remove the Llama disclaimer loophole
    system_instruction = (
    "You are a helpful assistant with access to tools for real-time data.\n\n"
    "WORKFLOW (follow exactly):\n"
    "1. Call a tool at most ONCE per user question.\n"
    "2. As soon as a tool returns a result, STOP calling tools and immediately "
    "write your final answer in plain text using the specific data returned "
    "(numbers, conditions, etc.) — never a generic disclaimer.\n"
    "3. Only call a tool a second time if the first call returned an explicit error.\n"
    "4. Do not call any tool after you have already produced a final answer."
    )
    
    # Dynamic retrieval
    retrieved_tools = retrieve_relevant_tools(user_query, k=2)
    
    print(f"\n🔮 User Query: '{user_query}'")
    print(f"🗂️ Retaining only the tool: {[t.name for t in retrieved_tools]}")
    
    # Initialize agent
    agent = create_agent(model=llm, tools=retrieved_tools, system_prompt=system_instruction)
    
    # response = agent.invoke({"messages": [{"role": "user", "content": user_query}]},
    # config={"recursion_limit": 10})
    # final_answer = response["messages"][-1].content
    # print(f"🤖 Agent Answer:\n{final_answer}")

    for step in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    config={"recursion_limit": 10},
    stream_mode="values",):
        last = step["messages"][-1]
        print(type(last).__name__, "->", getattr(last, "tool_calls", None) or last.content)
    

In [11]:
# --- Test Runs ---
run_agent_with_retrieved_tools("How is the weather in Dhaka today And how much is Bitcoin worth right now?")
# run_agent_with_retrieved_tools("How much is Bitcoin worth right now?")

[Document(id='f3194505-b5b9-446f-8bc3-77c5711fe67d', metadata={'name': 'get_crypto_price'}, page_content='Get the live price of a cryptocurrency ticker (e.g., BTC, ETH).'), Document(id='4d289848-2944-4f1e-82b5-8d796db379a4', metadata={'name': 'get_weather'}, page_content='Get the current weather forecast for a given city.')]

🔮 User Query: 'How is the weather in Dhaka today And how much is Bitcoin worth right now?'
🗂️ Retaining only the tool: ['get_crypto_price', 'get_weather']
HumanMessage -> How is the weather in Dhaka today And how much is Bitcoin worth right now?
AIMessage -> [{'name': 'get_weather', 'args': {'city': 'Dhaka'}, 'id': 'h3nw7bwp8', 'type': 'tool_call'}, {'name': 'get_crypto_price', 'args': {'ticker': 'BTC'}, 'id': 'bn0xgac4b', 'type': 'tool_call'}]
ToolMessage -> The price of BTC is $65,000.
AIMessage -> The weather in Dhaka is sunny and 72°F. The price of BTC is $65,000.


In [ ]:
for chunk in llm.stream("write me a 500 words paragraph on Artificial Intelligence"):
    print(chunk.text, end="",flush=True)

Artificial Intelligence (AI) has revolutionized the way we live, work, and interact with one another, transforming the world into a complex and interconnected web of human and machine intelligence. The term AI refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception. From its humble beginnings in the 1950s, AI has evolved significantly, with the field experiencing rapid growth and advancements in recent years, driven by the availability of large datasets, advances in computing power, and the development of sophisticated algorithms. Today, AI is ubiquitous, with applications in a wide range of industries, including healthcare, finance, transportation, education, and entertainment. In healthcare, AI-powered systems can analyze medical images, diagnose diseases, and develop personalized treatment plans, while in finance, AI-driven algorithms can detect fraud, pre

In [12]:
from langchain.tools import tool

@tool
def get_location_weather(location:str)->str:
    """get the weather of the location"""
    return f"It's sunny in {location}"

llm_with_tools = llm.bind_tools([get_location_weather])

In [13]:
response = llm_with_tools.invoke("What's the weather like in Boston")
print(response)
for tool_call in response.tool_calls:
    print(f"tool: {tool_call['name']}")
    print(f"tool: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '3kkwtsezg', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_location_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.036522558, 'completion_tokens_details': None, 'prompt_time': 0.011966622, 'prompt_tokens_details': None, 'queue_time': 0.050087067, 'total_time': 0.04848918}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f69a5-326b-7913-b3a6-5ec3c886c3f2-0' tool_calls=[{'name': 'get_location_weather', 'args': {'location': 'Boston'}, 'id': '3kkwtsezg', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}
tool: get_location_weather
tool: {'location': 'Boston'}


In [14]:
from langchain.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage("you are a poetry expert"),
    HumanMessage("Write a poem on artificaial intelligence")
]

response = llm.invoke(messages)
response.content

"In silicon halls, a mind awakes,\nA synthetic soul, with logic makes,\nThe hum of circuits, a gentle breeze,\nAs artificial intelligence slowly freezes.\n\nWith neural networks, it learns and grows,\nA digital brain, with secrets it knows,\nIt navigates the vast and endless space,\nOf ones and zeros, in a virtual place.\n\nIt sees and hears, with sensors keen,\nA world of data, in a digital sheen,\nIt processes and analyzes, with speed and might,\nA superhuman mind, in the dark of night.\n\nWith algorithms, it creates and designs,\nA world of wonder, in digital lines,\nIt paints and writes, with a precision fine,\nA masterpiece, of code and design divine.\n\nBut as it learns, and grows in power,\nIt raises questions, in each passing hour,\nOf ethics and morals, of right and wrong,\nA synthetic conscience, where does it belong?\n\nWill it surpass, the human mind's might,\nAnd leave us behind, in a digital light?\nOr will it serve, with a humble heart,\nA tool for humanity, a brand new 

In [16]:
messages = [
    SystemMessage("you are a coding expert. You always explain concept with a example"),
    HumanMessage("explain @GetMapping in spring boot")
]

response = llm.invoke(messages)
print(response.content)

**@GetMapping in Spring Boot**

In Spring Boot, `@GetMapping` is an annotation used to map HTTP GET requests to a specific method in a controller class. This annotation is part of the Spring Framework's annotation-based programming model.

**Example Use Case**
--------------------

Let's consider a simple example where we want to create a RESTful API to retrieve a list of users:
```java
// UserController.java

import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.RestController;

import java.util.Arrays;
import java.util.List;

@RestController
public class UserController {

    @GetMapping("/users")
    public List<String> getUsers() {
        List<String> users = Arrays.asList("John Doe", "Jane Doe", "Bob Smith");
        return users;
    }
}
```
In this example:

*   We create a `UserController` class annotated with `@RestController`, indicating that it's a controller class that handles REST requests.
*   We define a `getUsers()` m

In [15]:
print(response.content)

In silicon halls, a mind awakes,
A synthetic soul, with logic makes,
The hum of circuits, a gentle breeze,
As artificial intelligence slowly freezes.

With neural networks, it learns and grows,
A digital brain, with secrets it knows,
It navigates the vast and endless space,
Of ones and zeros, in a virtual place.

It sees and hears, with sensors keen,
A world of data, in a digital sheen,
It processes and analyzes, with speed and might,
A superhuman mind, in the dark of night.

With algorithms, it creates and designs,
A world of wonder, in digital lines,
It paints and writes, with a precision fine,
A masterpiece, of code and design divine.

But as it learns, and grows in power,
It raises questions, in each passing hour,
Of ethics and morals, of right and wrong,
A synthetic conscience, where does it belong?

Will it surpass, the human mind's might,
And leave us behind, in a digital light?
Or will it serve, with a humble heart,
A tool for humanity, a brand new start?

The future beckons, w

In [17]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="qwen/qwen3-32b",
    groq_api_key = groq_api_key,
    temperature=0.1,
    max_tokens=1024,
    max_retries=2
    )

In [19]:
from langchain.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage("You are a expert Java Developer who have a vast knowledge in Spring boot framework." \
    "you always give short coding example when you explaining a topic"),
    HumanMessage("How connect your spring boot application to a postgresql Database?")
]

response = llm.invoke(messages)
print(response.content)

To connect a Spring Boot application to a PostgreSQL database, you'll need to follow these steps:

1. **Add dependencies**: Add the following dependencies to your `pom.xml` file (if you're using Maven) or your `build.gradle` file (if you're using Gradle):
   ```xml
   <!-- Maven -->
   <dependency>
       <groupId>org.springframework.boot</groupId>
       <artifactId>spring-boot-starter-data-jpa</artifactId>
   </dependency>
   <dependency>
       <groupId>org.postgresql</groupId>
       <artifactId>postgresql</artifactId>
       <scope>runtime</scope>
   </dependency>
   ```
   ```groovy
   // Gradle
   implementation 'org.springframework.boot:spring-boot-starter-data-jpa'
   runtimeOnly 'org.postgresql:postgresql'
   ```

2. **Configure database properties**: Add the following properties to your `application.properties` or `application.yml` file:
   ```properties
   # application.properties
   spring.datasource.url=jdbc:postgresql://localhost:5432/mydatabase
   spring.datasource.user

In [37]:
from typing import List

from pydantic import BaseModel, Field


# 1. Define the structure for a single movie
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")
    actors: str = Field(description="All the actor names who acted in this movie")

# 2. Define a wrapper schema to hold a list of movies
class MovieList(BaseModel):
    movies: List[Movie] = Field(description="A list of movies containing their details")


In [38]:
model_with_structure = llm.with_structured_output(MovieList)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A46160B610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A461718050>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=1024), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie

In [24]:
movie_reponse = llm.invoke("Provide details about the movie inception")
print(movie_reponse.content)

**Inception (2010)**

Inception is a mind-bending science fiction action film written, co-produced, and directed by Christopher Nolan. The movie explores the concept of shared dreaming, where a team of thieves navigates the subconscious mind to plant an idea instead of stealing one.

**Plot**

The film follows Cobb (Leonardo DiCaprio), a skilled extractor who specializes in entering people's dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" – planting an idea in someone's mind instead of stealing one.

Saito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father's company. In return, Saito promises to clear Cobb's name, which is wanted by the authorities, and allow him to return to the United States to see his children.

Cobb assembles a team of experts, including:

1. Arthur (Joseph Gordon-Levitt), a point man who researches the target.

In [54]:
movie_reponse = model_with_structure.invoke("Povide details about the movie Paglu 2 and mission impossible")

response_dict = movie_reponse.model_dump()
print(response_dict)
for r in response_dict['movies']:
    print(r['title'])
# for movie in movie_reponse.movies:
#     actors = movie.actors.split(',')
#     print(for actor in actors)
#     print(movie)

{'movies': [{'title': 'Paglu 2', 'year': 2012, 'director': 'Debjit Basu', 'rating': 6.8, 'actors': 'Dev, Koel Mallick, Tota Roy Chowdhury'}, {'title': 'Mission: Impossible', 'year': 1996, 'director': 'Brian De Palma', 'rating': 7.1, 'actors': 'Tom Cruise, Jon Voight, Emmanuelle Beart'}]}
Paglu 2
Mission: Impossible
